# What it costs to build the oracle that holds the answer

A correction to one measurement, and nothing else. The scaling and spread sweeps in
`oracle_cost_lab.ipynb` stand; only its build-time cell was wrong, in two ways.

**It recorded nothing.** `marking_build_seconds` came back null for 15 of 22 boards. Not
failures: the writer tested `if marking_seconds`, which is false when the value is `0.0`, so
every sub-millisecond build was filed as a failure.

**And the thing it timed was not the thing it meant to time.** `PhaseOracleGate(expr)`
returns in about a millisecond at any size, because Qiskit builds the gate lazily. The
`2^n` truth-table reduction runs later, the first time something asks the gate what it
decomposes into. The old cell timed the constructor and concluded construction was free.

This notebook asks the gate for its `.definition`, which forces that work, and times it.
What comes back is the cost of reducing the formula over the whole grid space, with no
transpiler, no coupling map, no seeds and no device: the one number in this study that
depends on neither Qiskit's scheduler nor this machine's load.

It also records what the reduction produces. For a puzzle with `M` solutions the definition
should be `M` multi-controlled Z gates and nothing else, which is the claim that the oracle
encodes the answers rather than the question, in a form that can be checked rather than
argued.

## 1. Setup

In [ ]:
# pip install qiskit qiskit-algorithms qiskit-ibm-runtime
import json, os, signal, time

import qiskit

print("qiskit", qiskit.__version__)

CONFIG = {
    "min_cells": 4,
    "max_cells": 30,       # the sweep stops well before this on its own
    "cap_seconds": 900,    # stop once one board's reduction passes this
    "sample": 400,         # boards to draw per shape when looking for one to measure
    "sample_seed": 20260930,
}
OUT = "oracle-build-definition.jsonl"
CONFIG

## 2. The puzzle and oracle code

The same functions as `oracle_cost_lab.ipynb`, inline here so this notebook stands alone.
Nothing in this section changed.

In [ ]:
import itertools, math, random


def line_patterns(length, clue):
    '''Every filling of one line that matches its clue, as ints with bit 0 leftmost.'''
    if clue in ((0,), ()):
        return [0]
    out = []

    def place(block, start, bits):
        if block == len(clue):
            out.append(bits)
            return
        run = clue[block]
        rest = clue[block + 1:]
        room = sum(rest) + len(rest)
        for first in range(start, length - room - run + 1):
            mask = ((1 << run) - 1) << first
            place(block + 1, first + run + 1, bits | mask)

    place(0, 0, 0)
    return out


def line_clue(cells):
    '''The clue a filled line produces.'''
    out, run = [], 0
    for cell in cells:
        if cell:
            run += 1
        elif run:
            out.append(run)
            run = 0
    if run:
        out.append(run)
    return tuple(out) if out else (0,)


def clues_of(bits, rows, cols):
    grid = [bits[r * cols:(r + 1) * cols] for r in range(rows)]
    return ([line_clue(row) for row in grid],
            [line_clue([grid[r][c] for r in range(rows)]) for c in range(cols)])


def solve(row_clues, col_clues, limit=64):
    '''Every grid the clues accept, by placing a row at a time and pruning the columns.'''
    rows, cols = len(row_clues), len(col_clues)
    row_options = [line_patterns(cols, c) for c in row_clues]
    col_options = [line_patterns(rows, c) for c in col_clues]
    if any(not o for o in row_options) or any(not o for o in col_options):
        return []
    found = []

    def walk(r, placed, col_live):
        if len(found) >= limit:
            return
        if r == rows:
            found.append(list(placed))
            return
        for pattern in row_options[r]:
            nxt = []
            for c in range(cols):
                bit = (pattern >> c) & 1
                live = [p for p in col_live[c] if ((p >> r) & 1) == bit]
                if not live:
                    break
                nxt.append(live)
            else:
                placed.append(pattern)
                walk(r + 1, placed, nxt)
                placed.pop()

    walk(0, [], col_options)
    return found


def boards(rows, cols, sample, seed):
    '''Distinct solvable puzzles of this shape, drawn at random.'''
    seen = set()
    rng = random.Random(seed)
    for _ in range(sample):
        bits = [rng.randint(0, 1) for _ in range(rows * cols)]
        row_clues, col_clues = clues_of(bits, rows, cols)
        key = (tuple(row_clues), tuple(col_clues))
        if key in seen:
            continue
        seen.add(key)
        answers = solve(row_clues, col_clues)
        if answers:
            yield row_clues, col_clues, len(answers)


def lines_of(row_clues, col_clues):
    rows, cols = len(row_clues), len(col_clues)
    out = [([r * cols + c for c in range(cols)], tuple(row_clues[r])) for r in range(rows)]
    out += [([r * cols + c for r in range(rows)], tuple(col_clues[c])) for c in range(cols)]
    return out


def pattern_stats(row_clues, col_clues):
    '''What the clue oracle is built from, as numbers to plot the cost against.'''
    lines = lines_of(row_clues, col_clues)
    counts = [len(line_patterns(len(q), clue)) for q, clue in lines]
    widths = [len(q) for q, _ in lines]
    return {"lines": len(lines), "patterns": sum(counts),
            "mcx_width": sum(c * w for c, w in zip(counts, widths)),
            "longest_line": max(widths),
            "determined": sum(counts) == len(lines)}


def clue_oracle(row_clues, col_clues):
    '''A phase oracle that tests the clues: one flag qubit per line.'''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import MCXGate, ZGate

    cells = len(row_clues) * len(col_clues)
    lines = lines_of(row_clues, col_clues)
    qc = QuantumCircuit(cells + len(lines), name="clue oracle")

    def mark():
        for flag, (qubits, clue) in enumerate(lines):
            for pattern in line_patterns(len(qubits), clue):
                qc.append(MCXGate(len(qubits), ctrl_state=pattern), qubits + [cells + flag])

    mark()
    qc.append(ZGate().control(len(lines) - 1), [cells + i for i in range(len(lines))])
    mark()
    return qc


def clue_formula(row_clues, col_clues):
    '''The clues as a boolean expression, which is what Qiskit's oracle is built from.'''
    terms = []
    for qubits, clue in lines_of(row_clues, col_clues):
        options = []
        for pattern in line_patterns(len(qubits), clue):
            literals = [("v%d" % q if (pattern >> i) & 1 else "~v%d" % q)
                        for i, q in enumerate(qubits)]
            options.append("(" + " & ".join(literals) + ")")
        terms.append("(" + " | ".join(options) + ")")
    return " & ".join(terms)


def marking_oracle(row_clues, col_clues):
    '''Qiskit's oracle. The gate returns immediately; `.definition` does the work.'''
    from qiskit.circuit.library import PhaseOracleGate

    return PhaseOracleGate(clue_formula(row_clues, col_clues))


print("3x3 plus:", pattern_stats([(1,), (3,), (1,)], [(1,), (3,), (1,)]))

## 3. The measurement

Three timings per board, each taken separately:

| | what it covers |
|---|---|
| `clue_build` | assembling the clue oracle's circuit — one gate per allowed pattern |
| `marking_construct` | `PhaseOracleGate(expr)` returning. Lazy, so this is near zero at every size |
| `marking_define` | asking for `.definition`, which runs the `2^n` reduction |

`definition_size` and `definition_ops` record what the reduction emitted. For `M` solutions
expect `M` multi-controlled Z gates.

The sweep is serial and appends one line per board, so it can be interrupted and resumed. It
stops at the first board whose reduction passes `cap_seconds`, which is the point of the
exercise rather than a failure.

In [ ]:
def shapes_up_to(min_cells, max_cells):
    out = [(r, c) for r in range(2, max_cells + 1) for c in range(r, max_cells + 1)
           if min_cells <= r * c <= max_cells]
    return sorted(out, key=lambda rc: (rc[0] * rc[1], rc))


def done_keys(path):
    if not os.path.exists(path):
        return set()
    keys = set()
    with open(path) as handle:
        for line in handle:
            try:
                keys.add(json.loads(line)["key"])
            except Exception:
                pass
    return keys


def measure_build(row_clues, col_clues, solutions):
    '''One board, three timings. Every value is recorded, including a zero.'''
    started = time.time()
    clue = clue_oracle(row_clues, col_clues)
    clue_build = time.time() - started

    started = time.time()
    gate = marking_oracle(row_clues, col_clues)
    construct = time.time() - started

    started = time.time()
    definition = gate.definition
    define = time.time() - started

    ops = dict(definition.count_ops())
    return {
        "clue_build_seconds": round(clue_build, 6),
        "clue_gates": sum(clue.count_ops().values()),
        "marking_construct_seconds": round(construct, 6),
        "marking_define_seconds": round(define, 6),
        "definition_size": definition.size(),
        "definition_depth": definition.depth(),
        "definition_ops": ops,
        "definition_qubits": definition.num_qubits,
    }


def run(path=OUT):
    have = done_keys(path)
    for rows, cols in shapes_up_to(CONFIG["min_cells"], CONFIG["max_cells"]):
        key = "build:%dx%d" % (rows, cols)
        if key in have:
            continue
        board = None
        for row_clues, col_clues, solutions in boards(
                rows, cols, CONFIG["sample"], CONFIG["sample_seed"]):
            if not pattern_stats(row_clues, col_clues)["determined"]:
                board = (row_clues, col_clues, solutions)
                break
        if board is None:
            continue
        row_clues, col_clues, solutions = board

        record = {"key": key, "rows": rows, "cols": cols, "cells": rows * cols,
                  "solutions": solutions,
                  "row_clues": [list(c) for c in row_clues],
                  "col_clues": [list(c) for c in col_clues]}
        record.update(pattern_stats(row_clues, col_clues))
        try:
            record.update(measure_build(row_clues, col_clues, solutions))
        except Exception as exc:
            record["error"] = "%s: %s" % (type(exc).__name__, exc)

        with open(path, "a") as handle:
            handle.write(json.dumps(record) + "\n")

        if "error" in record:
            print("%2d cells  %s" % (record["cells"], record["error"]))
            continue
        print("%2d cells  M=%d  clue %7.4f s   marking construct %7.4f s   "
              "define %9.3f s   -> %d gate(s) %s"
              % (record["cells"], solutions, record["clue_build_seconds"],
                 record["marking_construct_seconds"], record["marking_define_seconds"],
                 record["definition_size"], list(record["definition_ops"])))

        if record["marking_define_seconds"] > CONFIG["cap_seconds"]:
            print("\npast the cap at %d cells; stopping." % record["cells"])
            return


run()

## 4. Read the results

In [ ]:
import re


def load(path=OUT):
    rows = []
    if os.path.exists(path):
        with open(path) as handle:
            for line in handle:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def marked_grids(ops):
    '''How many grids the definition marks: one multi-controlled Z each.

    The X gates around them set which grid each Z fires on, so they are addressing, not
    marking, and do not count.
    '''
    return sum(n for name, n in ops.items() if re.fullmatch(r"c\d+z(_o\d+)?", name))


rows = [r for r in load() if "marking_define_seconds" in r]
print("%4s %4s %3s %8s %10s %12s %8s %s"
      % ("n", "pats", "M", "clue s", "construct", "define s", "x clue", "marked"))
for r in sorted(rows, key=lambda r: r["cells"]):
    ratio = r["marking_define_seconds"] / max(r["clue_build_seconds"], 1e-9)
    print("%4d %4d %3d %8.4f %10.4f %12.3f %8.0f %6d"
          % (r["cells"], r["patterns"], r["solutions"], r["clue_build_seconds"],
             r["marking_construct_seconds"], r["marking_define_seconds"], ratio,
             marked_grids(r["definition_ops"])))

mismatch = [r for r in rows if marked_grids(r["definition_ops"]) != r["solutions"]]
if mismatch:
    print("\nboards where the definition does not mark one grid per solution:")
    for r in mismatch:
        print("  %s: M=%d, marked=%d, ops=%s"
              % (r["key"], r["solutions"], marked_grids(r["definition_ops"]),
                 r["definition_ops"]))
else:
    print("\nevery definition marks exactly one grid per solution, on %d boards" % len(rows))

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed; skipping the plot")

if plt and rows:
    rows_sorted = sorted(rows, key=lambda r: r["cells"])
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    ax.plot([r["cells"] for r in rows_sorted],
            [max(r["marking_define_seconds"], 1e-4) for r in rows_sorted],
            "o-", ms=4, lw=0.8, color="#c0392b", label="answer-marking: reduce 2^n rows")
    ax.plot([r["cells"] for r in rows_sorted],
            [max(r["marking_construct_seconds"], 1e-4) for r in rows_sorted],
            "s--", ms=3, lw=0.8, color="#c0392b", mfc="none",
            label="answer-marking: construct the gate")
    ax.plot([r["cells"] for r in rows_sorted],
            [max(r["clue_build_seconds"], 1e-4) for r in rows_sorted],
            "o-", ms=4, lw=0.8, color="#1a1a1a", label="clue oracle: build from the clues")
    ax.set_yscale("log")
    ax.set_xlabel("cells on the board")
    ax.set_ylabel("seconds")
    ax.set_title("Building each oracle, with no transpiler involved")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", lw=0.4, color="#dddddd")
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="upper left")
    plt.tight_layout()
    plt.show()

## 5. What to send back

`oracle-build-definition.jsonl`. Every record carries its own clue pair, solution count,
pattern statistics and all three timings, so it folds in without re-deriving anything.

Worth saying when you report it: the largest board the sweep reached, and whether it stopped
on the cap or ran out of shapes.